In [ ]:
!pip install -q ctranslate2 transformers==4.46.0 huggingface_hub tokenizers

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import ctranslate2
import os

MODEL_ID = "momosl/whisper-wolof-v1"
OUTPUT_DIR = "/content/whisper-wolof-ct2"
HF_REPO_CT2 = "momosl/whisper-wolof-v1-ct2"

print(f"Conversion de {MODEL_ID} vers CTranslate2...")
print("(~5 min)")

converter = ctranslate2.converters.TransformersConverter(
    MODEL_ID,
    copy_files=["tokenizer.json", "preprocessor_config.json", "special_tokens_map.json", "tokenizer_config.json", "added_tokens.json", "normalizer.json", "merges.txt", "vocab.json"]
)
converter.convert(
    OUTPUT_DIR,
    quantization="int8",
    force=True
)

print(f"\nConversion terminee ! Fichiers dans {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f} ({size:.1f} Mo)")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HF_REPO_CT2, exist_ok=True)

print(f"Push vers {HF_REPO_CT2}...")
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO_CT2,
    commit_message="Whisper Wolof v1 - CTranslate2 int8"
)

print(f"\n{'='*60}")
print(f"   MODELE CT2 PUBLIE !")
print(f"   https://huggingface.co/{HF_REPO_CT2}")
print(f"")
print(f"   Ce modele est compatible avec faster-whisper")
print(f"   et sera utilise par la Lambda AWS.")
print(f"{'='*60}")